In [1]:
# %% [markdown]
# # Week 3 — Day 21: Information Extraction Evaluation
#
# **Goal:** Verify that `extract_all()` works correctly on 30 CVs,
# measure skills extraction F1-score on 10 manually tagged CVs,
# log failure cases, and confirm we hit the target **F1 ≥ 0.75**.
#
# **Deliverables from this notebook:**
# - `data/processed/extracted_cvs.json` — 30 extracted CV objects
# - F1 score reported (target ≥ 0.75)
# - Failure case log printed at the bottom
#
# **Pipeline flow:**
# `cleaned_resumes.csv` → `extract_all()` → `CVSchema` → JSON

# %%
import os
import sys
import json
import warnings
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")

# 🛠️ FIX 1: Safely change working directory to project root if running inside notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../")

# Add the project root to sys.path
sys.path.append(os.getcwd())

from src.extractor.extractor import extract_all
from src.schema_validator import validate_cv, quick_check
from src.schema import CVSchema

# Paths (Now cleanly relative to the project root)
PROCESSED_DIR   = "data/processed/"
OUTPUT_JSON     = os.path.join(PROCESSED_DIR, "extracted_cvs.json")

# 🛠️ FIX 2: Point to your actual cleaned resume file here!
# Options: "structured_resumes_clean.csv", "ner_resumes_clean.csv", etc.
CLEANED_CSV     = os.path.join(PROCESSED_DIR, "datasetmaster_clean.csv")

print(f"✅ Working directory set to project root: {os.getcwd()}")
print(f"🎯 Target data file path: {CLEANED_CSV}")
# %% [markdown]
# ## Step 1 — Load 30 CVs from Week 2 output

# %%
try:
    df = pd.read_csv(CLEANED_CSV).tail(4500)
    print(f"Loaded {len(df)} CVs from {CLEANED_CSV}")
    print(f"Columns: {list(df.columns)}")
except FileNotFoundError:
    print("⚠️  cleaned_resumes.csv not found — using mock data for demo.")
    df = pd.DataFrame({
        "text": [
            "Jane Smith. jane@email.com | +880-171-000-0001\n"
            "EDUCATION\nB.Sc Computer Science, BUET, 2021, GPA: 3.75\n"
            "EXPERIENCE\nSoftware Engineer at Shohoz, Jan 2022 – Present\n"
            "Designed REST APIs using Django and PostgreSQL.\n"
            "SKILLS\nPython, Django, PostgreSQL, Docker, Git, React\n"
            "PROJECTS\nRide Tracking System | Tools: Python, Redis | github.com/jane/ride-tracker\n"
            "Built real-time tracking with 99.9% uptime.\n"
            "CERTIFICATIONS\nAWS Certified Developer – Associate, Amazon, 2023\n"
            "LANGUAGES\nEnglish (C1), Bengali (Native)\n"
            "LEADERSHIP\nTech Lead – BUET Programming Club 2020",
        ] * 30,
    })
    # Add mock individual section columns to mimic real CSV structure
    for col in ["personal_info", "experience", "education", "skills", "projects", "certifications", "achievements"]:
        df[col] = '{}'

# Build unified `sections` dict from individual CSV columns
# The extractor expects: {"education": "...", "experience": "...", "skills": "...", ...}
# Handle missing columns gracefully (structured_resumes_clean.csv may lack languages/leadership)

# section_cols = ["education", "experience", "skills", "projects", "certifications", "languages", "achievements", "leadership"]
# df["sections"] = df.apply(
#     lambda row: {col: str(row.get(col, "")) for col in section_cols},
#     axis=1
# )

# Build unified `sections` dict from individual CSV columns
section_cols = [
    "education", "experience", "skills", "projects", 
    "certifications", "languages", "achievements", "leadership", 
    "personal_info"  # 1. Added from overview to capture candidate summaries
]
df["sections"] = df.apply(
    lambda row: {col: str(row.get(col, "")) for col in section_cols},
    axis=1
)

✅ Working directory set to project root: d:\Projects\cvinsight
🎯 Target data file path: data/processed/datasetmaster_clean.csv
Loaded 4500 CVs from data/processed/datasetmaster_clean.csv
Columns: ['text', 'text_length', 'personal_info', 'experience', 'education', 'skills', 'projects', 'certifications', 'achievements']


In [2]:
# %%
extracted_cvs = []
failed_indices = []

for idx, row in df.iterrows():
    try:
        text = str(row.get("text", ""))
        sections = row.get("sections", {})
        
        # Execute master extractor string analysis
        cv_data = extract_all(text, sections=sections)
        extracted_cvs.append(cv_data)
    except Exception as e:
        failed_indices.append(idx)
        print(f"[FAIL] Row {idx}: {type(e).__name__}: {e}")

print(f"\n📊 Process complete: {len(extracted_cvs)} processed successfully, {len(failed_indices)} failed.")

# Save JSON array to disk
os.makedirs(PROCESSED_DIR, exist_ok=True)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(extracted_cvs, f, indent=2, ensure_ascii=False)
print(f"📁 Exported structured objects to: {OUTPUT_JSON}")


📊 Process complete: 4500 processed successfully, 0 failed.
📁 Exported structured objects to: data/processed/extracted_cvs.json


In [3]:
# %%
total = len(extracted_cvs)

if total == 0:
    print("❌ Cannot compute system overview metrics: 0 extractions succeeded.")
else:
    fields = ["name", "email", "phone", "education", "experience", "skills", "projects", "certifications", "languages", "achievements", "leadership"]
    coverage = {field: sum(1 for c in extracted_cvs if c.get(field)) for field in fields}
    
    coverage_df = pd.DataFrame([
        {"field": k, "populated_count": v, "coverage_%": round(v / total * 100, 1)}
        for k, v in coverage.items()
    ])
    print("📈 Data Extraction Field Coverage Density Map:")
    print(coverage_df.to_string(index=False))

📈 Data Extraction Field Coverage Density Map:
         field  populated_count  coverage_%
          name             4500       100.0
         email             4500       100.0
         phone             4500       100.0
     education             4500       100.0
    experience             4500       100.0
        skills             4500       100.0
      projects             4500       100.0
certifications                0         0.0
     languages             4500       100.0
  achievements                0         0.0
    leadership                0         0.0


In [8]:
for i,cv in enumerate(extracted_cvs):
    print(i,cv['name'] , cv['languages'])

0 Charles Owen [{'language': 'English', 'proficiency': None}]
1 Jerome Garcia [{'language': 'English', 'proficiency': None}]
2 Javier Garcia [{'language': 'English', 'proficiency': None}]
3 Michael Alexander [{'language': 'English', 'proficiency': None}]
4 Andre Ortiz [{'language': 'English', 'proficiency': None}]
5 Tyler Adams [{'language': 'English', 'proficiency': None}]
6 Michael Nichols [{'language': 'English', 'proficiency': None}]
7 Susan Hunt [{'language': 'English', 'proficiency': None}]
8 Melissa Weaver [{'language': 'English', 'proficiency': None}]
9 Rebecca Cooper DDS [{'language': 'English', 'proficiency': None}]
10 Roberto Phillips [{'language': 'English', 'proficiency': None}]
11 Julie Howard [{'language': 'English', 'proficiency': None}]
12 William Pacheco [{'language': 'English', 'proficiency': None}]
13 Rebecca Gomez [{'language': 'English', 'proficiency': None}]
14 Joshua Castaneda [{'language': 'English', 'proficiency': None}]
15 Pamela Anderson [{'language': 'Engli

In [5]:
# %%
print("📋 Evaluation Sample Hashing Keys (First 10 CVs):")
print("-" * 75)
for i, cv in enumerate(extracted_cvs[:10]):
    pred_preview = ", ".join(cv.get("skills", [])[:])
    print(f"Sample [{i}] ID: {cv['cv_id']} | Current Predicted Skills: {pred_preview or '(None Found)'}")

📋 Evaluation Sample Hashing Keys (First 10 CVs):
---------------------------------------------------------------------------
Sample [0] ID: 1592839a5a1c | Current Predicted Skills: c#, redis, mongodb, java, spring, ruby, english, react, spring boot, azure, javascript
Sample [1] ID: 36da9f14389c | Current Predicted Skills: flask, redis, go, java, spring, ruby, google cloud, english, spring boot
Sample [2] ID: 8ca4c60d286c | Current Predicted Skills: spring boot, java, spring, ruby, mysql, english, angular, azure, c++
Sample [3] ID: c195cde5c793 | Current Predicted Skills: c#, flask, mongodb, java, spring, english, django, node.js, spring boot, c++, aws, javascript
Sample [4] ID: 945bd3d0c5ef | Current Predicted Skills: vue, oracle, python, go, java, spring, english, postgresql, node.js, spring boot, c++, aws
Sample [5] ID: 673a746454bd | Current Predicted Skills: oracle, go, mongodb, java, spring, english, spring boot, azure, django
Sample [6] ID: 6308735e1f33 | Current Predicted Skills

In [6]:
# %%
# 🎯 POPULATED GROUND TRUTH MATRIX WITH YOUR REAL CV_IDS
# Review the original text fields for these 10 rows and list EVERY valid skill present.
manual_ground_truth = {
    "e8c8228c06f4": ["python", "c++", "c", "tensorflow", "numpy", "mysql"], 
    "bfd26ffd5363": ["project execution", "quality management", "budget monitoring", "plc programming", "scada systems", "sap", "microsoft visio", "english"],
    "674801e42f58": ["c", "sql", "pl/sql", "java", "javaee", "javascript", "html", "css", "jquery", "mysql", "spring", "hibernate", "python", "aws", "english"],
    "a969f392c418": ["python", "django", "mysql"],
    "7cf766ed2946": ["python"],
    "26a625190eb2": ["c", "c++", "j2ee", "spring", "hibernate", "my sql", "postgresql"], 
    "e36e1a4d0803": ["java", "spring", "hibernate", "mysql"],
    "3d20cc81b4d4": ["java", "sql", "pl/sql", "c", "c++", "jsp", "ext js", "oracle", "ms-sql", "ms-access", "ms-excel"], 
    "417fcd6ad988": ["java", "javascript"],
    "0c84c2b5d26d": ["c", "core java"]
}

def calculate_set_statistics(predicted: list, actual: list) -> tuple[float, float, float]:
    pred_set = set(str(s).lower().strip() for s in predicted if str(s).strip())
    true_set = set(str(s).lower().strip() for s in actual if str(s).strip())
    
    if not pred_set and not true_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not true_set:
        return 0.0, 0.0, 0.0
        
    true_positives = len(pred_set & true_set)
    precision = true_positives / len(pred_set)
    recall = true_positives / len(true_set)
    
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return round(precision, 3), round(recall, 3), round(f1, 3)

eval_records = []
for cv in extracted_cvs[:10]:
    cid = cv["cv_id"]
    pred_skills = cv.get("skills", [])
    actual_skills = manual_ground_truth.get(cid, [])
    
    p, r, f1 = calculate_set_statistics(pred_skills, actual_skills)
    
    pred_set = set(s.lower().strip() for s in pred_skills)
    actual_set = set(s.lower().strip() for s in actual_skills)
    
    eval_records.append({
        "cv_id": cid,
        "predicted_count": len(pred_skills),
        "actual_count": len(actual_skills),
        "precision": p,
        "recall": r,
        "f1_score": f1,
        "false_positives": ", ".join(sorted(pred_set - actual_set)) or "None",
        "false_negatives": ", ".join(sorted(actual_set - pred_set)) or "None"
    })

eval_df = pd.DataFrame(eval_records)
print("\n🎯 Precision, Recall, and F1 Performance Matrix:")
print(eval_df[["cv_id", "predicted_count", "actual_count", "precision", "recall", "f1_score"]].to_string(index=False))

mean_f1 = eval_df["f1_score"].mean()
print("\n" + "="*50)
print(f"📈 Combined Baseline System Mean F1 Score: {mean_f1:.3f}")
print("="*50)

if mean_f1 >= 0.75:
    print("✅ Target Achieved (F1 >= 0.75)! Pipeline validation complete. Proceed to Week 4.")
else:
    print("❌ Below Target. Inspect the isolated error gaps printed below to expand your skill taxonomy.")


🎯 Precision, Recall, and F1 Performance Matrix:
       cv_id  predicted_count  actual_count  precision  recall  f1_score
1592839a5a1c               11             0        0.0     0.0       0.0
36da9f14389c                9             0        0.0     0.0       0.0
8ca4c60d286c                9             0        0.0     0.0       0.0
c195cde5c793               12             0        0.0     0.0       0.0
945bd3d0c5ef               12             0        0.0     0.0       0.0
673a746454bd                9             0        0.0     0.0       0.0
6308735e1f33                8             0        0.0     0.0       0.0
5b030935d0e1               11             0        0.0     0.0       0.0
9e362f3f680b                9             0        0.0     0.0       0.0
6e10b283e557               11             0        0.0     0.0       0.0

📈 Combined Baseline System Mean F1 Score: 0.000
❌ Below Target. Inspect the isolated error gaps printed below to expand your skill taxonomy.


In [7]:
# %%
print("🔍 System Discrepancy & Diagnostic Log:\n")
for rec in eval_records:
    if rec["f1_score"] < 1.0:
        print(f"📋 Profile Hash ID: {rec['cv_id']} (F1: {rec['f1_score']})")
        print(f"  🔻 Missed (False Negatives): {rec['false_negatives']}")
        print(f"  🔺 Hallucinated (False Positives): {rec['false_positives']}\n")

🔍 System Discrepancy & Diagnostic Log:

📋 Profile Hash ID: 1592839a5a1c (F1: 0.0)
  🔻 Missed (False Negatives): None
  🔺 Hallucinated (False Positives): azure, c#, english, java, javascript, mongodb, react, redis, ruby, spring, spring boot

📋 Profile Hash ID: 36da9f14389c (F1: 0.0)
  🔻 Missed (False Negatives): None
  🔺 Hallucinated (False Positives): english, flask, go, google cloud, java, redis, ruby, spring, spring boot

📋 Profile Hash ID: 8ca4c60d286c (F1: 0.0)
  🔻 Missed (False Negatives): None
  🔺 Hallucinated (False Positives): angular, azure, c++, english, java, mysql, ruby, spring, spring boot

📋 Profile Hash ID: c195cde5c793 (F1: 0.0)
  🔻 Missed (False Negatives): None
  🔺 Hallucinated (False Positives): aws, c#, c++, django, english, flask, java, javascript, mongodb, node.js, spring, spring boot

📋 Profile Hash ID: 945bd3d0c5ef (F1: 0.0)
  🔻 Missed (False Negatives): None
  🔺 Hallucinated (False Positives): aws, c++, english, go, java, node.js, oracle, postgresql, python, sp